In [ ]:
import os
import json
import pandas as pd
import streamlit as st
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# -----------------------------
# PAGE CONFIGURATION
# -----------------------------

st.set_page_config(
    page_title="AI Expense Tracker",
    page_icon="💰",
    layout="wide"
)

st.title("💰 AI Expense Tracker")
st.write("Track your expenses and use AI to understand your spending.")

# -----------------------------
# AI CLIENT
# -----------------------------

api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
else:
    client = None


# -----------------------------
# SESSION STORAGE
# -----------------------------

if "expenses" not in st.session_state:
    st.session_state.expenses = []


# -----------------------------
# AI CATEGORY FUNCTION
# -----------------------------

def categorize_expense(description):

    if not client:
        return "Other"

    prompt = f"""
You are a personal finance assistant.

Your task is to categorize an expense.

Allowed categories:
- Food
- Transport
- Shopping
- Education
- Bills
- Entertainment
- Healthcare
- Other

Examples:

Expense: "I bought pizza for ₹300"
Category: Food

Expense: "I paid ₹200 for an Uber"
Category: Transport

Expense: "I purchased a Python book for ₹500"
Category: Education

Now categorize this expense:

Expense: {description}

Return only the category name.
"""

    try:

        response = client.responses.create(
            model="gpt-5.6-mini",
            input=prompt
        )

        category = response.output_text.strip()

        allowed = [
            "Food",
            "Transport",
            "Shopping",
            "Education",
            "Bills",
            "Entertainment",
            "Healthcare",
            "Other"
        ]

        if category in allowed:
            return category

        return "Other"

    except Exception as e:
        st.error(f"AI error: {e}")
        return "Other"


# -----------------------------
# ADD EXPENSE
# -----------------------------

st.header("➕ Add Expense")

description = st.text_input(
    "Expense description",
    placeholder="Example: I spent ₹250 on lunch"
)

amount = st.number_input(
    "Amount (₹)",
    min_value=0.0,
    step=10.0
)

date = st.date_input("Date")

if st.button("🤖 Add Expense"):

    if not description:
        st.warning("Please enter an expense description.")

    elif amount <= 0:
        st.warning("Please enter an amount greater than zero.")

    else:

        with st.spinner("AI is categorizing your expense..."):

            category = categorize_expense(description)

        expense = {
            "Description": description,
            "Amount": amount,
            "Category": category,
            "Date": str(date)
        }

        st.session_state.expenses.append(expense)

        st.success(
            f"Expense added successfully! AI category: {category}"
        )


# -----------------------------
# DISPLAY EXPENSES
# -----------------------------

st.header("📋 Your Expenses")

if st.session_state.expenses:

    df = pd.DataFrame(st.session_state.expenses)

    st.dataframe(
        df,
        use_container_width=True,
        hide_index=True
    )

else:

    st.info("No expenses added yet.")


# -----------------------------
# DASHBOARD
# -----------------------------

if st.session_state.expenses:

    st.header("📊 Expense Dashboard")

    df = pd.DataFrame(st.session_state.expenses)

    total = df["Amount"].sum()

    count = len(df)

    average = total / count

    col1, col2, col3 = st.columns(3)

    with col1:
        st.metric(
            "Total Expenses",
            f"₹{total:.2f}"
        )

    with col2:
        st.metric(
            "Transactions",
            count
        )

    with col3:
        st.metric(
            "Average Expense",
            f"₹{average:.2f}"
        )

    # Category totals

    st.subheader("Category-wise Spending")

    category_totals = (
        df.groupby("Category")["Amount"]
        .sum()
        .sort_values(ascending=False)
    )

    st.bar_chart(category_totals)


# -----------------------------
# DELETE EXPENSES
# -----------------------------

if st.session_state.expenses:

    st.header("🗑️ Delete Expense")

    expense_numbers = list(
        range(len(st.session_state.expenses))
    )

    selected = st.selectbox(
        "Select expense",
        expense_numbers,
        format_func=lambda x:
            f"{x + 1}. {st.session_state.expenses[x]['Description']}"
    )

    if st.button("Delete Selected Expense"):

        st.session_state.expenses.pop(selected)

        st.success("Expense deleted.")

        st.rerun()


# -----------------------------
# AI FINANCIAL SUMMARY
# -----------------------------

if st.session_state.expenses:

    st.header("🤖 AI Spending Assistant")

    if st.button("Generate AI Spending Summary"):

        if not client:

            st.warning(
                "Add your OpenAI API key to enable AI analysis."
            )

        else:

            expense_data = json.dumps(
                st.session_state.expenses,
                indent=2
            )

            summary_prompt = f"""
You are a personal finance assistant.

Analyze the following expense data.

{expense_data}

Provide:

1. Total spending
2. Highest spending category
3. Two observations about the spending
4. Three simple suggestions for saving money

Keep the response short and easy to understand.
"""

            try:

                with st.spinner(
                    "AI is analyzing your expenses..."
                ):

                    response = client.responses.create(
                        model="gpt-5.6-mini",
                        input=summary_prompt
                    )

                st.write(response.output_text)

            except Exception as e:

                st.error(f"AI error: {e}")

ModuleNotFoundError: No module named 'streamlit'